# 99. Presentation Figures

Generates all data-driven figures for the lab meeting slides.
Set `SAVE = True` to export PNGs to `figures/`.

Slide numbers refer to the agreed structure:
- 7: What distinguishes BE from SC
- 9-10: Validation examples
- 11: SBI posteriors
- 12-13: Cohort accuracy + assignment strip
- 14: Parameter recovery
- 16: Real data assignment strip
- 17-21: Example animals (SS05, SS07)
- 22-23: Parameter distributions + GS vs SBI
- 24: Dynamic SBI trajectories

In [ ]:
%matplotlib inline
from shared_setup import *
import pickle, torch
from pathlib import Path
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from plotting.animal_report import (
    plot_animal_summary, plot_cv_results,
    plot_model_fits, plot_sbi_diagnostics,
    _get_sbi_samples, _load_gs_raw_pickles, _gs_seed_errors,
)
from plotting.validation_report import (
    plot_synth_summary, plot_synth_cv_results,
    plot_synth_model_fits, plot_synth_sbi_diagnostics,
    build_gs_index, extract_gs_recovery, extract_sbi_recovery,
    plot_recovery_overlay, plot_recovery_summary, build_confusion_matrix
)
from plotting.sbi import plot_parameter_trajectories
from analysis.consensus import load_all_assignments, consensus_summary
from behav_utils.plotting.styles import COLOURS

BE_COL, SC_COL = COLOURS['BE'], COLOURS['SC']

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
SAVE = False  # Set True to export PNGs
FIG_DIR = RESULTS_DIR.parent / 'figures' / 'presentation'
if SAVE:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Saving figures to {FIG_DIR}')

EXAMPLE_ANIMALS = ['SS05', 'SS07']
COHORTS = ['static_uniform', 'learning_uniform']

_fig_counter = [0]
_original_show = plt.show

def _saving_show(*args, **kwargs):
    """Hook: save current figure before showing."""
    if SAVE:
        fig = plt.gcf()
        _fig_counter[0] += 1
        path = FIG_DIR / f'fig_{_fig_counter[0]:02d}.png'
        fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    _original_show(*args, **kwargs)

plt.show = _saving_show

## Load All Data

In [ ]:
# Real data
experiment, info = load_data()
snpe = load_snpe_networks()
assign_df = load_all_assignments(RESULTS_DIR, experiment)
print(consensus_summary(assign_df))

# Synthetic cohorts
cohort_data = {}
for cohort in COHORTS:
    p = VALIDATION_DIR / 'synthetic_cohorts' / f'{cohort}.pkl'
    if p.exists():
        with open(p, 'rb') as f:
            cohort_data[cohort] = pickle.load(f)['animals']
        print(f'Cohort {cohort}: {len(cohort_data[cohort])} animals')

# GS raw pickles + summaries
synth_gs_raw, synth_gs_dfs = {}, {}
for cohort in COHORTS:
    for ft in FIT_TARGETS:
        d = VALIDATION_DIR / 'synth_gs' / f'{cohort}_{ft}'
        if not d.exists(): continue
        sp = d / 'summary.pkl'
        if sp.exists():
            with open(sp, 'rb') as f:
                synth_gs_dfs[(cohort, ft)] = pickle.load(f)['df']
        raws = []
        for p in sorted(d.glob('synth_*.pkl')):
            if p.name == 'summary.pkl': continue
            with open(p, 'rb') as f: raws.append(pickle.load(f))
        if raws: synth_gs_raw[(cohort, ft)] = raws

# SBI pickles
synth_sbi_dfs = {}
for cohort in COHORTS:
    for ft in FIT_TARGETS:
        d = VALIDATION_DIR / 'synth_sbi' / f'{cohort}_{ft}'
        if not d or not d.exists(): continue
        files = sorted(d.glob('synth_*.pkl'))
        if files:
            rows = []
            for p in files:
                with open(p, 'rb') as f: rows.append(pickle.load(f))
            synth_sbi_dfs[(cohort, ft)] = pd.DataFrame(rows)

gs_index = build_gs_index(synth_gs_raw)

In [ ]:
# Pick synthetic examples
examples = {}
for i, sa in enumerate(cohort_data.get('static_uniform', [])):
    if i == 0:
        continue
    mt = sa['true_model']
    if mt in examples: continue
    aid = sa['animal_id']
    has_both = all(
        ft in gs_index and aid in gs_index[ft]
        and 'BE' in gs_index[ft][aid] and 'SC' in gs_index[ft][aid]
        for ft in FIT_TARGETS
    )
    if has_both:
        examples[mt] = sa
    if len(examples) == 2: break

for mt, sa in examples.items():
    print(f'Synthetic example {mt}: {sa["animal_id"]}')

---
## Slide 7: What Distinguishes BE from SC

Synthetic examples: identical psychometrics, different update matrices.

In [ ]:
for mt in ['BE', 'SC']:
    if mt not in examples: continue
    plot_synth_summary(examples[mt])

---
## Slides 9-10: Validation — Example BE and SC

GS violins + UM comparisons for one true-BE and one true-SC animal.

In [ ]:
for mt in ['BE', 'SC']:
    if mt not in examples: continue
    sa = examples[mt]
    print(f'\n{"═" * 50}')
    print(f'  {sa["animal_id"]} (true {mt})')
    print(f'{"═" * 50}')
    for ft in FIT_TARGETS:
        fig = plot_synth_cv_results(sa, gs_index, fit_target=ft)
        fig = plot_synth_model_fits(sa, gs_index, fit_target=ft)

---
## Slide 11: SBI Posteriors for Example Animal

Posterior marginals + GS points + truth.

In [ ]:
for mt in ['BE', 'SC']:
    if mt not in examples: continue
    sa = examples[mt]
    plot_synth_sbi_diagnostics(sa, snpe, gs_index=gs_index, show_ppc=False)

---
## Slide 12: Cohort Accuracy + Confusion Matrices

In [ ]:
# Accuracy table
acc_rows = []
for cohort in COHORTS:
    for ft in FIT_TARGETS:
        row = {'cohort': cohort, 'fit_target': FT_LABEL[ft]}
        key = (cohort, ft)
        for prefix, src in [('gs', synth_gs_dfs), ('sbi', synth_sbi_dfs)]:
            if key in src:
                df = src[key]
                col = 'correct' if 'correct' in df.columns else f'{prefix}_correct'
                if col in df.columns:
                    row[f'{prefix}_acc'] = df[col].mean()
                    row[f'{prefix}_nc'] = int(df[col].sum())
                    row[f'{prefix}_n'] = len(df)
        acc_rows.append(row)
acc_df = pd.DataFrame(acc_rows)

# Accuracy bars
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
ft_colours = {'UM': BE_COL, 'CP': SC_COL}
for ax_i, (method, ac, nc, nn) in enumerate([
    ('Grid Search', 'gs_acc', 'gs_nc', 'gs_n'),
    ('SBI (SNPE)', 'sbi_acc', 'sbi_nc', 'sbi_n'),
]):
    ax = axes[ax_i]
    sub = acc_df.dropna(subset=[ac])
    if len(sub) == 0:
        ax.text(0.5, 0.5, f'No {method} results', transform=ax.transAxes, ha='center')
        ax.set_title(method); continue
    cohorts_u = sub['cohort'].unique()
    x = np.arange(len(cohorts_u))
    fts = sub['fit_target'].unique()
    w = 0.35
    for i, ft in enumerate(fts):
        fs = sub[sub['fit_target'] == ft]
        vals, lbls = [], []
        for c in cohorts_u:
            r = fs[fs['cohort'] == c]
            if len(r): vals.append(r[ac].values[0]); lbls.append(f'{int(r[nc].values[0])}/{int(r[nn].values[0])}')
            else: vals.append(0); lbls.append('')
        bars = ax.bar(x+(-w/2+w*i), vals, w*0.9, label=ft, color=ft_colours.get(ft, f'C{i}'), alpha=0.8)
        for bar, lbl in zip(bars, lbls):
            if lbl: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, lbl, ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([c.replace('_','\n') for c in cohorts_u], fontsize=9)
    ax.axhline(0.5, color='grey', ls='--', lw=0.8)
    ax.set_ylim(0, 1.15); ax.set_ylabel('Accuracy' if ax_i==0 else '')
    ax.set_title(method, fontsize=12, fontweight='bold'); ax.legend(fontsize=8, loc='lower right')
fig.suptitle('Model Identification Accuracy', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
cms = {}
for cohort in COHORTS:
    for ft in FIT_TARGETS:
        lc = cohort.replace('_',' ').title()
        key = (cohort,ft)
        if key in synth_gs_dfs:
            cm = build_confusion_matrix(synth_gs_dfs[key], true_col='true_model', pred_col='winner')
            if cm is not None: cms[f'GS-{FT_LABEL[ft]}\n{lc}'] = cm
        if key in synth_sbi_dfs:
            cm = build_confusion_matrix(synth_sbi_dfs[key], true_col='true_model', pred_col='winner')
            if cm is not None: cms[f'SBI-{FT_LABEL[ft]}\n{lc}'] = cm

if cms:
    n = len(cms)
    fig, axes = plt.subplots(1,n,figsize=(3.2*n,3.8)); axes = np.atleast_1d(axes)
    for ax,(title,cm) in zip(axes,cms.items()):
        cm_n = cm/cm.sum(axis=1,keepdims=True)
        ax.imshow(cm_n,cmap='Blues',vmin=0,vmax=1,aspect='equal')
        for i in range(2):
            for j in range(2):
                c = 'white' if cm_n[i,j]>0.5 else 'black'
                ax.text(j,i,f'{cm[i,j]}\n({cm_n[i,j]*100:.0f}%)',ha='center',va='center',fontsize=12,fontweight='bold',color=c)
        ax.set_xticks([0,1]);ax.set_xticklabels(['BE','SC'])
        ax.set_yticks([0,1]);ax.set_yticklabels(['BE','SC'])
        ax.set_xlabel('Assigned');ax.set_ylabel('True')
        tot=cm.sum(); ax.set_title(f'{title}\n{cm.trace()}/{tot} ({cm.trace()/tot:.0%})',fontsize=9,fontweight='bold')
        for s in ax.spines.values(): s.set_visible(False)
    fig.suptitle('Confusion Matrices',fontsize=13,fontweight='bold')
    plt.tight_layout(); plt.show()

---
## Slide 14: Parameter Recovery

In [ ]:
gs_um_recovery = extract_gs_recovery(
    synth_gs_raw.get(('static_uniform', 'update_matrix'), []))
gs_cp_recovery = extract_gs_recovery(
    synth_gs_raw.get(('static_uniform', 'conditional_psych'), []))

if 'static_uniform' in cohort_data and len(snpe) == 2:
    print('Re-conditioning SNPE on all synthetic animals...')
    sbi_recovery = extract_sbi_recovery(cohort_data['static_uniform'], snpe)
else:
    sbi_recovery = {'BE': {}, 'SC': {}}

plot_recovery_overlay(gs_um_recovery, gs_cp_recovery, sbi_recovery)
plot_recovery_summary(gs_um_recovery, gs_cp_recovery, sbi_recovery)

---
## Slide 16: Real Data — Cohort Assignment Strip

In [ ]:
# Reuse the strip plot from NB 15 section 1
method_cols = ['GS-UM', 'GS-CP', 'SBI-UM', 'SBI-CP']
all_cols = method_cols + ['Consensus']
n_methods = len(all_cols)
n_animals = len(assign_df)

fig, ax = plt.subplots(figsize=(1.5 * n_methods + 2, max(6, n_animals * 0.38)))
colour_map = {'BE': BE_COL, 'SC': SC_COL}

for i, (_, row) in enumerate(assign_df.iterrows()):
    for j, col in enumerate(all_cols):
        val = row.get(col)
        p_val = row.get(f'{col}_p', np.nan) if col != 'Consensus' else np.nan
        if val in ('BE', 'SC'):
            fc = colour_map[val]
            alpha = 0.9 if (col == 'Consensus' or (pd.notna(p_val) and p_val < 0.05)) else 0.25
            rect = plt.Rectangle((j-0.45, i-0.45), 0.9, 0.9,
                                 facecolor=fc, alpha=alpha, edgecolor='black', lw=0.5)
            ax.add_patch(rect)
            label = val + (f'\n{p_val:.3f}' if col != 'Consensus' and pd.notna(p_val) else '')
            ax.text(j, i, label, ha='center', va='center', fontsize=7,
                    fontweight='bold', color='white' if alpha > 0.5 else 'black')
        elif val == 'Split':
            ax.add_patch(plt.Rectangle((j-0.45,i-0.45),0.9,0.9, facecolor='#FFE0B2', edgecolor='black', lw=0.5))
            ax.text(j, i, 'Split', ha='center', va='center', fontsize=7, fontweight='bold', color='#E65100')
        elif val == 'Unclear':
            ax.add_patch(plt.Rectangle((j-0.45,i-0.45),0.9,0.9, facecolor='#F0F0F0', edgecolor='#CCC', lw=0.5))
            ax.text(j, i, '?', ha='center', va='center', fontsize=9, color='#999')
        else:
            ax.add_patch(plt.Rectangle((j-0.45,i-0.45),0.9,0.9, facecolor='#F8F8F8', edgecolor='#DDD', lw=0.5))
            ax.text(j, i, '—', ha='center', va='center', fontsize=9, color='#BBB')

ax.axvline(n_methods - 1.5, color='black', lw=2.5)
ax.set_xlim(-0.6, n_methods - 0.4); ax.set_ylim(-0.6, n_animals - 0.4)
ax.set_xticks(range(n_methods)); ax.set_xticklabels(all_cols, fontsize=10, fontweight='bold')
ax.xaxis.set_ticks_position('top')
ax.set_yticks(range(n_animals)); ax.set_yticklabels(assign_df['id'].values, fontsize=8)
ax.invert_yaxis()
ax.legend(handles=[
    Patch(facecolor=BE_COL, alpha=0.9, label='BE (p<0.05)'),
    Patch(facecolor=BE_COL, alpha=0.25, label='BE (not sig.)'),
    Patch(facecolor=SC_COL, alpha=0.9, label='SC (p<0.05)'),
    Patch(facecolor=SC_COL, alpha=0.25, label='SC (not sig.)'),
    Patch(facecolor='#FFE0B2', edgecolor='#E65100', label='Split'),
    Patch(facecolor='#D4D4D4', label='Inconclusive'),
], loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8)
cons_counts = assign_df['Consensus'].value_counts()
cons_str = ', '.join(f'{k}: {v}' for k, v in cons_counts.items())
ax.set_title(f'Model Assignment — All Animals\n{cons_str}', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

---
## Slides 17-21: Example Animals (SS05 + SS07)

In [ ]:
for AID in EXAMPLE_ANIMALS:
    print(f'\n{"═" * 70}')
    print(f'  {AID}')
    print(f'{"═" * 70}')

    if AID not in experiment.animals:
        print(f'{AID} not found'); continue

    sessions = select_sessions(experiment.animals[AID], 'expert_uniform')
    if not sessions:
        print('No expert uniform sessions'); continue

    # Get assign_row for this animal (avoids re-loading all pickles)
    match = assign_df[assign_df['id'] == AID]
    ar = match.iloc[0] if len(match) > 0 else None

    # Raw behaviour + consensus
    plot_animal_summary(AID, sessions, RESULTS_DIR, assign_row=ar)

    # GS CV — both fit targets
    for ft in FIT_TARGETS:
        plot_cv_results(AID, RESULTS_DIR, fit_target=ft)

    # Model fits
    plot_model_fits(AID, sessions, RESULTS_DIR, method='sbi')

    # SBI diagnostics
    plot_sbi_diagnostics(AID, sessions, snpe, RESULTS_DIR)

---
## Slide 22: Parameter Distributions Across Cohort

In [ ]:
if len(snpe) == 2:
    param_data = []
    for aid in sorted(experiment.animals.keys()):
        sessions = select_sessions(experiment.animals[aid], 'expert_uniform')
        if len(sessions) < 3: continue
        consensus = assign_df.loc[assign_df['id'] == aid, 'Consensus'].values
        consensus = consensus[0] if len(consensus) > 0 else 'Unclear'
        be_samples, sc_samples, _ = _get_sbi_samples(
            aid, sessions, RESULTS_DIR, snpe, 'uniform', n_samples=5000)
        for mk, samples in [('be', be_samples), ('sc', sc_samples)]:
            if samples is None: continue
            pnames = snpe[mk]['param_names']
            for j, pn in enumerate(pnames):
                param_data.append({
                    'animal_id': aid, 'model': mk.upper(), 'param': pn,
                    'median': np.median(samples[:, j]),
                    'ci_lo': np.percentile(samples[:, j], 5),
                    'ci_hi': np.percentile(samples[:, j], 95),
                    'consensus': consensus,
                })
    param_df = pd.DataFrame(param_data)
    print(f'{param_df["animal_id"].nunique()} animals')

    for mt in ['BE', 'SC']:
        sub = param_df[param_df['model'] == mt]
        params = sub['param'].unique()
        n = len(params)
        fig, axes = plt.subplots(1, n, figsize=(4*n, 5)); axes = np.atleast_1d(axes)
        for i, pn in enumerate(params):
            ax = axes[i]
            psub = sub[sub['param'] == pn].sort_values('animal_id')
            for j, (_, row) in enumerate(psub.iterrows()):
                col = BE_COL if row['consensus']=='BE' else SC_COL if row['consensus']=='SC' else 'grey'
                ax.errorbar(j, row['median'],
                            yerr=[[row['median']-row['ci_lo']], [row['ci_hi']-row['median']]],
                            fmt='o', color=col, ms=5, capsize=3, elinewidth=1, alpha=0.8)
            ax.set_xticks(range(len(psub)))
            ax.set_xticklabels(psub['animal_id'].values, rotation=45, ha='right', fontsize=7)
            ax.set_ylabel(pn); ax.set_title(pn, fontsize=11)
        fig.suptitle(f'{mt} — SBI Parameter Estimates', fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

---
## Slide 23: GS vs SBI Consistency

In [ ]:
consistency_data = {'BE': {}, 'SC': {}}
for aid in sorted(experiment.animals.keys()):
    sessions = select_sessions(experiment.animals[aid], 'expert_uniform')
    if len(sessions) < 3: continue
    be_data, sc_data = _load_gs_raw_pickles(aid, RESULTS_DIR, 'uniform', 'update_matrix')
    if not be_data or not sc_data: continue
    be_samples, sc_samples, _ = _get_sbi_samples(
        aid, sessions, RESULTS_DIR, snpe, 'uniform', n_samples=5000)
    for model_name, gs_data, samples in [
        ('BE', be_data, be_samples), ('SC', sc_data, sc_samples)
    ]:
        _, gs_params = _gs_seed_errors(gs_data)
        if not gs_params or samples is None: continue
        mk = model_name.lower()
        if mk not in snpe: continue
        pnames = snpe[mk]['param_names']
        for j, pn in enumerate(pnames):
            if pn not in gs_params: continue
            if pn not in consistency_data[model_name]:
                consistency_data[model_name][pn] = {'gs': [], 'sbi': [], 'aid': []}
            consistency_data[model_name][pn]['gs'].append(gs_params[pn])
            consistency_data[model_name][pn]['sbi'].append(np.median(samples[:, j]))
            consistency_data[model_name][pn]['aid'].append(aid)

for mt in ['BE', 'SC']:
    params = consistency_data.get(mt, {})
    if not params: continue
    pnames = list(params.keys())
    n = len(pnames)
    col = BE_COL if mt == 'BE' else SC_COL
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5)); axes = np.atleast_1d(axes)
    for i, pn in enumerate(pnames):
        ax = axes[i]
        gs_vals = np.array(params[pn]['gs'])
        sbi_vals = np.array(params[pn]['sbi'])
        ax.scatter(gs_vals, sbi_vals, color=col, s=40, alpha=0.7, edgecolors='black', lw=0.5)
        for g, s, a in zip(gs_vals, sbi_vals, params[pn]['aid']):
            ax.annotate(a, (g, s), fontsize=6, alpha=0.6, xytext=(3, 3), textcoords='offset points')
        all_v = np.concatenate([gs_vals, sbi_vals])
        lo, hi = all_v.min(), all_v.max()
        m = (hi - lo) * 0.1
        lims = [lo - m, hi + m]
        ax.plot(lims, lims, 'k--', alpha=0.3)
        r = np.corrcoef(gs_vals, sbi_vals)[0, 1] if len(gs_vals) > 2 else np.nan
        ax.text(0.05, 0.95, f'r = {r:.3f}\nn = {len(gs_vals)}',
                transform=ax.transAxes, fontsize=9, va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax.set(xlabel=f'GS-UM {pn}', ylabel=f'SBI {pn}', aspect='equal')
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_title(pn, fontsize=11)
    fig.suptitle(f'{mt} — GS vs SBI Parameter Consistency', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

---
## Slide 24: Dynamic SBI — Parameters Stable During Expert Phase

In [ ]:
DYN_DIR = SBI_DYNAMIC_DIR / 'uniform_update_matrix'

if DYN_DIR.exists():
    for AID in EXAMPLE_ANIMALS:
        for model in ['BE', 'SC']:
            p = DYN_DIR / f'{AID}_{model}.pkl'
            if not p.exists(): continue
            with open(p, 'rb') as f: d = pickle.load(f)
            print(f'{AID} {model} ({d["n_sessions"]} sessions)')
            fig = plot_parameter_trajectories(
                d['trajectories'],
                title=f'{AID} — {model} Parameter Trajectories '
                      f'(Expert phase, {d["n_sessions"]} sessions)',
                show_samples=20,
            )
            plt.show()
else:
    print('No dynamic SBI results found')

---
## Done

Restore `plt.show` and report.

In [ ]:
plt.show = _original_show
if SAVE:
    figs = sorted(FIG_DIR.glob('fig_*.png'))
    print(f'\n{len(figs)} figures saved to {FIG_DIR}')
    for f in figs:
        print(f'  {f.name}')
else:
    print('SAVE is False — set SAVE = True and re-run to export PNGs.')